# 트랜스포머의 그래디언트 흐름 - 잔차 연결과 그래디언트 고속도로

- Tutorial ID: `adv-3-1`
- Tutorial: 트랜스포머의 그래디언트 흐름
- Section ID: `adv-3-1-1`
- Section: 잔차 연결과 그래디언트 고속도로


In [ ]:
# ============================================================
# 코드 읽는 법 — 잔차 연결과 그래디언트 고속도로
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) logit이 확률분포로 바뀌는 과정과 temperature/top-k/top-p의 효과 관찰
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np

print("=" * 60)
print("그래디언트 흐름과 잔차 연결")
print("=" * 60)

In [ ]:
# 1. 잔차 연결 없이 vs 있을 때

In [ ]:
print("\n1. 잔차 연결의 효과")
print("-" * 45)

n_layers = 50

# 잔차 없이: 그래디언트 곱
grad_no_residual = 1.0
for i in range(n_layers):
    grad_no_residual *= 0.25  # sigmoid 최대 미분

# 잔차 있을 때
np.random.seed(42)
grad_samples = []
for _ in range(1000):
    grad = 1.0
        for i in range(n_layers):
        dF_dx = 0.1 * np.random.randn()
        grad *= (1 + dF_dx)
    grad_samples.append(grad)

print(f"{n_layers}개 레이어 후 그래디언트:")
print(f"  잔차 없음 (sigmoid): {grad_no_residual:.2e} (소실!)")
print(f"  잔차 있음 (평균): {np.mean(grad_samples):.4f}")
print(f"  잔차 있음 [10%,90%]: [{np.percentile(grad_samples,10):.4f}, {np.percentile(grad_samples,90):.4f}]")
print(f"  → 잔차 연결이 그래디언트를 보존!")

In [ ]:
# 2. 소프트맥스의 야코비안

In [ ]:
print("\n2. 소프트맥스의 그래디언트 분석")
print("-" * 45)

def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum()

def softmax_jacobian(s):
    n = len(s)
    J = np.zeros((n, n))
        for i in range(n):
                for j in range(n):
            if i == j:
                J[i, j] = s[i] * (1 - s[i])
            else:
                J[i, j] = -s[i] * s[j]
    return J

scores = np.array([2.0, 1.0, 0.5, 0.0])
attn = softmax(scores)
J = softmax_jacobian(attn)

print(f"어텐션 점수: {scores}")
print(f"어텐션 가중치: {np.round(attn, 4)}")
print(f"야코비안 노름: {np.linalg.norm(J, 'fro'):.4f}")

print("\n온도별 그래디언트:")
for temp in [0.5, 1.0, 2.0, 5.0]:
    a = softmax(scores * temp)
    J_t = softmax_jacobian(a)
    entropy = -np.sum(a * np.log(a + 1e-10))
    print(f"  1/T={temp}: 엔트로피={entropy:.3f}, ||J||={np.linalg.norm(J_t, 'fro'):.4f}")
print("  → 집중될수록 그래디언트 작아짐!")

In [ ]:
# 3. 그래디언트 클리핑

In [ ]:
print("\n3. 그래디언트 클리핑")
print("-" * 45)

def clip_grad_norm(grads, max_norm):
    total_norm = np.sqrt(sum(np.sum(g**2) for g in grads))
    clip_coef = min(1.0, max_norm / (total_norm + 1e-6))
    clipped = [g * clip_coef for g in grads]
    print(f"  노름: {total_norm:.4f} → {total_norm * clip_coef:.4f} (max={max_norm})")
    return clipped

np.random.seed(0)
grads = [np.random.randn(100, 100) * 10, np.random.randn(50) * 10]
clip_grad_norm(grads, max_norm=1.0)

In [ ]:
# 4. 학습률 워밍업

In [ ]:
print("\n4. 학습률 워밍업")
print("-" * 45)

def lr_schedule(step, d_model=512, warmup=4000):
    return d_model ** (-0.5) * min(step ** (-0.5), step * warmup ** (-1.5))

for step in [1, 100, 1000, 4000, 10000]:
    lr = lr_schedule(step)
    print(f"  step {step:5d}: lr = {lr:.6f}")

print("\n워밍업 이유: 초기 그래디언트 방향 불안정 → 작은 lr로 시작")